# Xeno Data Analyst Assignment — Comm-Log Reconciliation

Finance says the target_base for merchant 501 in October 2026 is 22, I’ll start with the raw data and try to figure out how we get to that number.

In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("comm_log.db")

## Getting the tables

In [8]:
pd.read_sql_query(
    """
    select name from sqlite_master where type='table';
    """, conn
)

,name
0,campaign
1,communication_log


In [9]:
pd.read_sql_query(
    """
    select * from campaign; """, conn
)

,id,merchant_id,parent_id,name,creation_status,processing_status
0,9001,501,NaN,Diwali Cart Recovery - Wave 1,approved,processed
1,9002,501,9001.0,Diwali Cart Recovery - Retry A,approved,processed
2,9003,501,9002.0,Diwali Cart Recovery - Retry B,approved,processed
3,9004,501,9001.0,Diwali Cart Recovery - Retry C (pending),approval_awaiting,processed
4,9101,501,NaN,Diwali Flash Sale - Standalone,approved,processed
5,9201,501,NaN,Diwali Wave 2,approved,processed
6,9202,501,9201.0,Diwali Wave 2 - Retry,approved,processed


In [10]:
pd.read_sql_query(
    """
    select * from communication_log;""", conn
)

,id,merchant_id,communication_id,customer_id,communication_type,delivery_status,sent_time,scheduled_time,credit_used,channel
0,1,501,9001,C1,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
1,2,501,9001,C2,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
2,3,501,9002,C2,2,900,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
3,4,501,9001,C3,2,1100,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
4,5,501,9002,C3,2,1100,2026-10-04 10:00:00,2026-10-04 10:00:00,1,sms
5,6,501,9003,C3,2,900,2026-10-05 10:00:00,2026-10-05 10:00:00,1,sms
6,7,501,9001,C4,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
7,8,501,9001,C5,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
8,9,501,9001,C6,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms
9,10,501,9001,C7,2,900,2026-10-03 10:00:00,2026-10-03 10:00:00,1,sms


## Checking the data scope

Before starting the count, I'll check whether the data matches the time range given in the assignment.

In [30]:
pd.read_sql_query(
    """
     select merchant_id, min(sent_time) as first_send, max(sent_time) as last_send, communication_type
    from communication_log
    group by merchant_id, communication_type;
    """,
    conn
)

,merchant_id,first_send,last_send,communication_type
0,501,2026-10-03 10:00:00,2026-10-20 10:00:00,2


## Checking delivery statuses

I also checked the different delivery status values in the log before using them in the analysis.

In [31]:
pd.read_sql_query(
    """
    select delivery_status, count(*) as send_count
    from communication_log
    group by delivery_status
    order by delivery_status;
    """,
    conn
)

,delivery_status,send_count
0,900,26
1,1100,4


## Checking for duplicate records

In [33]:
pd.read_sql_query(
    """
    select communication_id,  customer_id, delivery_status, sent_time, count(*) as row_count
    from communication_log
    group by communication_id, customer_id, delivery_status, sent_time
    having count(*) > 1;
    """,
    conn
)

,communication_id,customer_id,delivery_status,sent_time,row_count


## number of all send records

In [11]:
pd.read_sql_query(
    """
    select count(*) as total_count
    from communication_log; """,conn)

,total_count
0,30


Total count came to be 30 but the finance's reported 22. I'll get the unique customers out of these.

In [12]:
pd.read_sql_query(
    """
    select count(distinct customer_id) as unique_customers 
    from communication_log;
    """,
    conn
)

,unique_customers
0,25


This shows that there total 25 unique customers, I'll look into the repeated customers to know the actual difference

In [13]:
pd.read_sql_query(
    """
    select customer_id,count(*) as send_count
    from communication_log
    group by customer_id
    having count(*) > 1
    order by send_count desc, customer_id;
    """,
    conn
)

,customer_id,send_count
0,C3,3
1,C2,2
2,C20,2
3,D1,2


I will dig deeper to find why there were repeated send commands for these customers

In [14]:
pd.read_sql_query(
    """
    select
        customer_id,
        communication_id,
        delivery_status,
        sent_time
    from communication_log
    where customer_id in ('C2', 'C3', 'C20', 'D1')
    order by customer_id, sent_time;
    """,
    conn
)

,customer_id,communication_id,delivery_status,sent_time
0,C2,9001,1100,2026-10-03 10:00:00
1,C2,9002,900,2026-10-04 10:00:00
2,C20,9101,900,2026-10-10 10:00:00
3,C20,9101,900,2026-10-20 10:00:00
4,C3,9001,1100,2026-10-03 10:00:00
5,C3,9002,1100,2026-10-04 10:00:00
6,C3,9003,900,2026-10-05 10:00:00
7,D1,9201,1100,2026-10-07 10:00:00
8,D1,9202,900,2026-10-08 10:00:00


from the readme provided to me I know, 900 = delivered and 1100 = not delivered/failed, some repeated customers have different campaign id's. checking the reason behind that

In [15]:
pd.read_sql_query(
    """
    select id as campaign_id, parent_id,name
    from campaign
    where parent_id is not null
    order by id;
    """,
    conn
)

,campaign_id,parent_id,name
0,9002,9001,Diwali Cart Recovery - Retry A
1,9003,9002,Diwali Cart Recovery - Retry B
2,9004,9001,Diwali Cart Recovery - Retry C (pending)
3,9202,9201,Diwali Wave 2 - Retry


This gives away that repeated customers have retry relationships, I'll check the sends within them.

In [17]:
pd.read_sql_query(
    """
    select communication_id, customer_id, delivery_status, sent_time
    from communication_log
    where communication_id in (9001, 9002, 9003)
    order by customer_id, sent_time;
    """,
    conn
)

,communication_id,customer_id,delivery_status,sent_time
0,9001,C1,900,2026-10-03 10:00:00
1,9001,C10,900,2026-10-03 10:00:00
2,9001,C2,1100,2026-10-03 10:00:00
3,9002,C2,900,2026-10-04 10:00:00
4,9001,C3,1100,2026-10-03 10:00:00
5,9002,C3,1100,2026-10-04 10:00:00
6,9003,C3,900,2026-10-05 10:00:00
7,9001,C4,900,2026-10-03 10:00:00
8,9001,C5,900,2026-10-03 10:00:00
9,9001,C6,900,2026-10-03 10:00:00


The 9001 to 9003 campaigns contain repeated attempts for C2 and C3. I’ll compare the number of send records with the number of customers in this chain.

In [18]:
pd.read_sql_query(
    """
    select count(*) as send_count, count(distinct customer_id) as customer_count
    from communication_log
    where communication_id in (9001, 9002, 9003);
    """,
    conn
)

,send_count,customer_count
0,13,10


This retry chain has more send records than customers because some customers were sent again. I’ll check the other retry chain to see if the same pattern exists there as well.

In [19]:
pd.read_sql_query(
    """
    select communication_id, customer_id, delivery_status, sent_time
    from communication_log
    where communication_id in (9201, 9202)
    order by customer_id, sent_time;
    """,
    conn
)

,communication_id,customer_id,delivery_status,sent_time
0,9201,D1,1100,2026-10-07 10:00:00
1,9202,D1,900,2026-10-08 10:00:00
2,9201,D2,900,2026-10-07 10:00:00
3,9201,D3,900,2026-10-07 10:00:00
4,9201,D4,900,2026-10-07 10:00:00
5,9201,D5,900,2026-10-07 10:00:00


This also shows same pattern, D1 was sent again after the first attempt failed

In [20]:
pd.read_sql_query(
    """
    select communication_id, customer_id, delivery_status, sent_time
    from communication_log
    where communication_id = 9101
    order by customer_id, sent_time;
    """,
    conn
)

,communication_id,customer_id,delivery_status,sent_time
0,9101,C20,900,2026-10-10 10:00:00
1,9101,C20,900,2026-10-20 10:00:00
2,9101,C21,900,2026-10-10 10:00:00
3,9101,C22,900,2026-10-10 10:00:00
4,9101,C23,900,2026-10-10 10:00:00
5,9101,C24,900,2026-10-10 10:00:00
6,9101,C25,900,2026-10-10 10:00:00


C20 was sent twice under the same campaign, 9101, on different dates even thought the delivery status shows 'delivered', from the readme we know 9101 isnt a retry chain, its a standalone campaign. So the both the sent commands should be counted in this case. 

In [21]:
pd.read_sql_query(
    """
    select count(*) as send_count, count(distinct customer_id) as customer_count
    from communication_log
    where communication_id = 9101;
    """,
    conn
)

,send_count,customer_count
0,7,6


Even though the unique customer count is 6 but we'll consider all the send count because its an standalone campaign.

## Looking back at the campaign data again

The repeated send issue seems to be explained by the retry relationships. I also noticed that the campaigns don't all have the same creation status, so I'll check whether that makes a difference to the count.

In [22]:
pd.read_sql_query(
    """
    select creation_status, processing_status, count(*) as campaign_count
    from campaign
    group by creation_status, processing_status
    order by creation_status, processing_status;
    """,
    conn
)

,creation_status,processing_status,campaign_count
0,approval_awaiting,processed,1
1,approved,processed,6


There is one campaign with 'approval_waiting' status, i checked the readme says this means the campaign has not cleared approval yet and campaigns in this state do not count toward reported sends. I’ll check how many communication records are linked to this campaign.

In [23]:
pd.read_sql_query(
    """
    select communication_id, count(*) as send_count
    from communication_log
    where communication_id = 9004
    group by communication_id;
    """,
    conn
)

,communication_id,send_count
0,9004,4


There are 4 communication records for 9004 campaign and as these have 'approval_waiting' status these will not be considered. Therefore excluding these at the earliest for the target_base

In [24]:
pd.read_sql_query(
    """
    select count(*) as eligible_send_count
    from communication_log cl
    join campaign c
        on cl.communication_id = c.id
    where c.creation_status in ('approved', 'aborted', 'resumed', 'stopped')
      and c.processing_status = 'processed';
    """,
    conn
)

,eligible_send_count
0,26


## Going back to the retry chains

We have already established that 26 send records remain after excluding the campaign that was still awaiting approval. I’ll now go back to the retry chains we identified earlier and calculate how many of those 26 records are repeated attempts, starting with 9001 - 9002 - 9003.

In [25]:
pd.read_sql_query(
    """
    select
        count(*) as send_count,
        count(distinct customer_id) as customer_count,
        count(*) - count(distinct customer_id) as repeated_sends
    from communication_log
    where communication_id IN (9001, 9002, 9003);
    """,
    conn
)

,send_count,customer_count,repeated_sends
0,13,10,3


The 9001 - 9002 - 9003 chain has 13 send records but only 10 customers. This means 3 records are additional attempts for customers already included in the chain. I’ll subtract these 3 from the 26 reportable records and then check the next campaign chain.

In [26]:
pd.read_sql_query(
    """
    select
        26 - (
            count(*) - count(distinct customer_id)
        ) as remaining_count
    from communication_log
    where communication_id in (9001, 9002, 9003);
    """,
    conn
)

,remaining_count
0,23


The count is now 23 after substracting the 3 repeated attempts in the first retry chain. I’ll now check the second retry chain, 9201 - 9202, and make the same adjustment if needed.

In [27]:
pd.read_sql_query(
    """
    select
        count(*) as send_count,
        count(distinct customer_id) as customer_count,
        count(*) - count(distinct customer_id) as repeated_sends
    from communication_log
    where communication_id in (9201, 9202);
    """,
    conn
)

,send_count,customer_count,repeated_sends
0,6,5,1


In [28]:
pd.read_sql_query(
    """
    select
        23 - (count(*) - count(distinct customer_id)) as final_count
    from communication_log
    where communication_id IN (9201, 9202);
    """,
    conn
)

,final_count
0,22


## Final result

After excluding the 4 records from campaign 9004 and accounting for the repeated attempts in the two retry chains, the final target_base is 22, which matches Finance's number.

In [34]:
reconciliation = pd.DataFrame({
    "Step": [
        "Starting count",
        "Unique customer check",
        "Exclude campaign 9004",
        "Retry chain 9001 → 9002 → 9003",
        "Retry chain 9201 → 9202",
        "Final"
    ],
    "Adjustment": [
        0,
        0,
        -4,
        -3,
        -1,
        0
    ],
    "Result": [
        30,
        25,
        26,
        23,
        22,
        22
    ]
})

reconciliation

,Step,Adjustment,Result
0,Starting count,0,30
1,Unique customer check,0,25
2,Exclude campaign 9004,-4,26
3,Retry chain 9001 → 9002 → 9003,-3,23
4,Retry chain 9201 → 9202,-1,22
5,Final,0,22
